<a href="https://colab.research.google.com/github/ishwarikale44-byte/Financial-Analytics-Project/blob/main/Copy_of_BT4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install web3 eth-tester py-evm py-solc-x

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.6/798.6 kB 39.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.3/586.3 kB 26.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [eth-tester]


In [ ]:
from web3 import Web3, EthereumTesterProvider
from solcx import compile_source, install_solc

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [ ]:
install_solc("0.8.20")

print("Solidity Compiler Installed Successfully")

Solidity Compiler Installed Successfully


In [ ]:
student_code = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract StudentData {

    struct Student {
        uint rollNo;
        string name;
        uint marks;
    }

    Student[] public students;

    function addStudent(
        uint _rollNo,
        string memory _name,
        uint _marks
    ) public {
        students.push(Student(_rollNo, _name, _marks));
    }

    function getStudent(uint index)
        public view returns(uint, string memory, uint)
    {
        Student memory s = students[index];
        return (s.rollNo, s.name, s.marks);
    }

    function getCount() public view returns(uint) {
        return students.length;
    }

    fallback() external payable {
    }
}
'''

print("Student Smart Contract Created Successfully")

Student Smart Contract Created Successfully


In [ ]:
compiled_sol = compile_source(
    student_code,
    solc_version="0.8.20"
)

contract_interface = list(compiled_sol.values())[0]

bytecode = contract_interface['bin']
abi = contract_interface['abi']

print("Smart Contract Compiled Successfully")

Smart Contract Compiled Successfully


In [ ]:
w3 = Web3(EthereumTesterProvider())

account = w3.eth.accounts[0]

w3.eth.default_account = account

print("Blockchain Connected Successfully")
print("Account Address:", account)

Blockchain Connected Successfully
Account Address: 0x7E5F4552091A69125d5DfCb7b8C2659029395Bdf


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
Student = w3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

tx_hash = Student.constructor().transact()

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

student_contract = w3.eth.contract(
    address=receipt.contractAddress,
    abi=abi
)

print("Smart Contract Deployed Successfully")
print("Contract Address:", student_contract.address)

Smart Contract Deployed Successfully
Contract Address: 0xF2E246BB76DF876Cef8b38ae84130F4F55De395b


In [ ]:
tx = student_contract.functions.addStudent(
    1, "Rahul", 85
).transact()

receipt1 = w3.eth.wait_for_transaction_receipt(tx)

print("Student Added Successfully")

Student Added Successfully


In [ ]:
tx = student_contract.functions.addStudent(
    2, "Priya", 90
).transact()

receipt2 = w3.eth.wait_for_transaction_receipt(tx)

print("Second Student Added Successfully")

Second Student Added Successfully


In [ ]:
count = student_contract.functions.getCount().call()

print("Total Students:", count)

for i in range(count):
    data = student_contract.functions.getStudent(i).call()

    print("Roll No:", data[0])
    print("Name:", data[1])
    print("Marks:", data[2])
    print("----------------")

Total Students: 2
Roll No: 1
Name: Rahul
Marks: 85
----------------
Roll No: 2
Name: Priya
Marks: 90
----------------


In [ ]:
tx = w3.eth.send_transaction({
    'from': account,
    'to': student_contract.address,
    'data': '0x',
    'value': 0
})

fallback_receipt = w3.eth.wait_for_transaction_receipt(tx)

print("Fallback Function Executed Successfully")

Fallback Function Executed Successfully


In [ ]:
gas_used = receipt1.gasUsed

gas_price = receipt1.effectiveGasPrice

fee = gas_used * gas_price

print("----- TRANSACTION DETAILS -----")
print("Gas Used:", gas_used)
print("Gas Price:", Web3.from_wei(gas_price, 'gwei'), "Gwei")
print("Transaction Fee:", Web3.from_wei(fee, 'ether'), "ETH")

----- TRANSACTION DETAILS -----
Gas Used: 112396
Gas Price: 1 Gwei
Transaction Fee: 0.000112396 ETH


In [ ]:
print("===== STUDENT DATA MANAGEMENT =====")

print("Total Students:",
      student_contract.functions.getCount().call())

print("Student 1:",
      student_contract.functions.getStudent(0).call())

print("Student 2:",
      student_contract.functions.getStudent(1).call())

print("Gas Used:", receipt1.gasUsed)

print("Transaction Fee:",
      Web3.from_wei(
          receipt1.gasUsed * receipt1.effectiveGasPrice,
          'ether'
      ), "ETH")

print("Status: Successful")

===== STUDENT DATA MANAGEMENT =====
Total Students: 2
Student 1: [1, 'Rahul', 85]
Student 2: [2, 'Priya', 90]
Gas Used: 112396
Transaction Fee: 0.000112396 ETH
Status: Successful
